# TypePro Python shard 05 of 10

Settings required: **Internet ON**, accelerator **None/CPU**. This
standalone notebook is permanently assigned shard `05` and
publishes `typepro-build-shard-05` as a
`public` Dataset under
`duymign`.

The notebook uses its same-account Kaggle host identity by default.
Optional target-owner Secrets `TYPEPRO_PUBLISH_USERNAME` and
`TYPEPRO_PUBLISH_KEY` (or legacy `KAGGLE_USERNAME` and `KAGGLE_KEY`)
override it. The notebook refuses to run when the effective publish
owner is not the expected owner.


In [ ]:
# This standalone notebook permanently builds exactly one shard.
ASSIGNED_SHARDS = [5]
SHARD_INDEX = 5
SHARD_COUNT = 10
EXPECTED_DATASET_OWNER = 'duymign'
PUBLISH_PUBLIC = True
REPOSITORY = 'https://github.com/duyvu1105/TypePro.git'
BRANCH = 'main'
SEED = 13
TEST_PROJECTS = 100
VALIDATION_PROJECT_RATIO = 0.10
SLICE_LOG_EVERY = 50
# Trace every annotation so the parent hard-kill timer is armed for
# each one; a pathological slice in native code cannot stall a shard.
SLICE_TRACE_EVERY = 1
# Bound every annotation so one pathological slice cannot stall a shard.
SLICE_ANNOTATION_TIMEOUT_SECONDS = 120
# Bound work that happens before annotation export as well.
PACKAGE_DOWNLOAD_TIMEOUT_SECONDS = 30
KB_PHASE_TIMEOUT_SECONDS = 300
PROJECT_ANALYSIS_TIMEOUT_SECONDS = 300
# Kill an exporter stuck building a project index after 30 minutes.
SLICE_INDEX_TIMEOUT_SECONDS = 1800
# Kill a stuck git clone after 15 minutes.
CLONE_TIMEOUT_SECONDS = 900
# This rerun keeps built-in annotations and adds function returns.
INCLUDE_BUILTINS = True
INCLUDE_RETURNS = True
RETRIEVAL_SCHEMA_VERSION = "typepro-project-kb-top10-generative-v2"

from pathlib import Path

REPO_DIR = Path("/kaggle/working/TypePro")
WORK_DIR = Path(f"/kaggle/working/typepro_build_shard_{SHARD_INDEX:02d}")
PUBLISH_DIR = Path(f"/kaggle/working/publish_shard_{SHARD_INDEX:02d}")
if SHARD_INDEX not in ASSIGNED_SHARDS:
    raise RuntimeError(
        f"Shard {SHARD_INDEX} is not assigned to this notebook: {ASSIGNED_SHARDS}"
    )
print({
    "assigned_shards": ASSIGNED_SHARDS,
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "expected_dataset_owner": EXPECTED_DATASET_OWNER or None,
    "publish_public": PUBLISH_PUBLIC,
    "work_dir": str(WORK_DIR),
})


## Authenticate safely

Optional values are read from Kaggle Secrets and are never printed. If
supplied, a separate config directory prevents the notebook host's
automatic credential from overriding the explicit credential.


In [ ]:
import json
import os
import re
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def optional_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        return None
    value = value.strip() if value else ""
    return value or None

target_username = optional_secret("TYPEPRO_PUBLISH_USERNAME")
target_key = optional_secret("TYPEPRO_PUBLISH_KEY")
legacy_username = optional_secret("KAGGLE_USERNAME")
legacy_key = optional_secret("KAGGLE_KEY")
if bool(target_username) != bool(target_key):
    raise RuntimeError(
        "TYPEPRO_PUBLISH_USERNAME and TYPEPRO_PUBLISH_KEY must both be present"
    )
if bool(legacy_username) != bool(legacy_key):
    raise RuntimeError("KAGGLE_USERNAME and KAGGLE_KEY must both be present")
publish_username = target_username or legacy_username
publish_key = target_key or legacy_key
use_explicit_credential = bool(publish_username and publish_key)
if not use_explicit_credential:
    # The notebook is pushed to the same account that owns its shard
    # Datasets. Kaggle supplies that host identity automatically, so
    # Secrets are optional for this normal same-account path.
    publish_username = EXPECTED_DATASET_OWNER
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{1,49}", publish_username):
    raise ValueError(f"Invalid publish username: {publish_username!r}")
if (
    EXPECTED_DATASET_OWNER
    and publish_username.casefold() != EXPECTED_DATASET_OWNER.casefold()
):
    raise RuntimeError(
        f"Publish credential belongs to {publish_username!r}; all TypePro "
        f"Datasets must belong to {EXPECTED_DATASET_OWNER!r}"
    )

if use_explicit_credential:
    auth_config_dir = Path("/kaggle/working/typepro_publish_auth")
    auth_config_dir.mkdir(parents=True, exist_ok=True)
    os.environ["KAGGLE_CONFIG_DIR"] = str(auth_config_dir)
    os.environ["KAGGLE_USERNAME"] = publish_username
    os.environ["KAGGLE_KEY"] = publish_key
    os.environ.pop("KAGGLE_API_TOKEN", None)
    authentication_source = "explicit Kaggle Secret"
else:
    authentication_source = "Kaggle notebook host"
# publish_shard.py validates the requested Dataset owner against this
# non-secret value. Keep the host access token untouched.
os.environ["KAGGLE_USERNAME"] = publish_username
os.environ["PYTHONUNBUFFERED"] = "1"
print({
    "publish_owner": publish_username,
    "expected_owner": EXPECTED_DATASET_OWNER or publish_username,
    "authentication_source": authentication_source,
    "credentials_printed": False,
})


## Clone TypePro and install builder dependencies


In [ ]:
import shutil
import subprocess
import sys

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, REPO_DIR])
else:
    print("Using existing repository:", REPO_DIR)

PIPELINE_DIR = REPO_DIR / "codet5p_type_retrieval"
run([sys.executable, "-m", "pip", "install", "-q", "-U", "-r", PIPELINE_DIR / "requirements-build.txt"])
if use_explicit_credential:
    # This release always honors the explicit legacy username/key pair.
    run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "kaggle==1.7.4.2"])
else:
    # Current Kaggle releases understand the notebook host's automatic
    # access token. No credential value is copied into this notebook.
    run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"])

identity_code = "\n".join([
    "import json",
    "import kaggle",
    "values = getattr(kaggle.api, 'config_values', {})",
    "print('TYPEPRO_KAGGLE_IDENTITY=' + json.dumps({",
    "    'username': values.get('username'),",
    "    'auth_method': values.get('auth_method') or 'LEGACY_API_KEY',",
    "}))",
])
identity_result = subprocess.run(
    [sys.executable, "-c", identity_code],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
identity_prefix = "TYPEPRO_KAGGLE_IDENTITY="
identity_line = next(
    (
        line for line in (identity_result.stdout or "").splitlines()
        if line.startswith(identity_prefix)
    ),
    None,
)
if identity_result.returncode or identity_line is None:
    raise RuntimeError(
        f"Kaggle publish authentication failed: {identity_result.stdout}"
    )
identity = json.loads(identity_line[len(identity_prefix):])
authenticated_username = (identity.get("username") or "").strip()
if authenticated_username.casefold() != publish_username.casefold():
    raise RuntimeError(
        f"Kaggle authenticated as {authenticated_username!r}, expected "
        f"publish owner {publish_username!r}"
    )
print({
    "authenticated_publish_owner": authenticated_username,
    "authentication_method": identity.get("auth_method"),
})


## Optional automatic resume

If the private shard dataset already exists, its archive is downloaded
and restored before slicing. A missing dataset simply means this is the
first run.


In [ ]:
import json
import zipfile

dataset_id = f"{publish_username}/typepro-build-shard-{SHARD_INDEX:02d}"
resume_dir = Path(f"/kaggle/working/resume_shard_{SHARD_INDEX:02d}")
resume_dir.mkdir(parents=True, exist_ok=True)
probe = subprocess.run(
    ["kaggle", "datasets", "files", dataset_id],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
has_work = (
    (WORK_DIR / "metadata" / "split_manifest.json").exists()
    or any((WORK_DIR / "raw_slices").glob("*.jsonl"))
    or any((WORK_DIR / "project_status").glob("*.json"))
)
if probe.returncode == 0 and not has_work:
    run(["kaggle", "datasets", "download", "-d", dataset_id, "-p", resume_dir, "--unzip"])
    markers = list(resume_dir.rglob("shard_manifest.json"))
    if not markers:
        archives = list(resume_dir.rglob("typepro_build_shard_*.zip"))
        if len(archives) != 1:
            raise RuntimeError(
                f"Expected one shard directory or archive, found markers={markers}, archives={archives}"
            )
        with zipfile.ZipFile(archives[0]) as bundle:
            bundle.extractall(resume_dir)
        markers = list(resume_dir.rglob("shard_manifest.json"))
    if len(markers) != 1:
        raise RuntimeError(f"Cannot uniquely locate restored shard: {markers}")
    source_build = markers[0].parent
    restored_manifest = json.loads(markers[0].read_text(encoding="utf-8"))
    restored_runtime_path = source_build / "runtime_manifest.json"
    restored_runtime = (
        json.loads(restored_runtime_path.read_text(encoding="utf-8"))
        if restored_runtime_path.exists() else {}
    )
    if (
        restored_manifest.get("shard_index") != SHARD_INDEX
        or restored_manifest.get("shard_count") != SHARD_COUNT
        or restored_runtime.get("retrieval_schema_version")
        != RETRIEVAL_SCHEMA_VERSION
    ):
        print(
            "Ignoring an incompatible previous shard Dataset; starting fresh:",
            {
                "manifest": restored_manifest,
                "retrieval_schema_version": restored_runtime.get(
                    "retrieval_schema_version"
                ),
                "expected_retrieval_schema_version": RETRIEVAL_SCHEMA_VERSION,
            },
        )
    else:
        shutil.copytree(source_build, WORK_DIR, dirs_exist_ok=True)
        print("Restored previous shard state:", WORK_DIR)
else:
    print("Starting new shard or using current working state")


## Download metadata and create the deterministic project split


In [ ]:
prepare = PIPELINE_DIR / "prepare_dataset.py"
common = [
    "--typepro-root", REPO_DIR,
    "--work-dir", WORK_DIR,
    "--split-profile", "paper_project",
    "--test-projects", TEST_PROJECTS,
    "--validation-project-ratio", VALIDATION_PROJECT_RATIO,
    "--seed", SEED,
    "--preview-samples", 1,
    "--preview-max-chars", 1200,
]
if INCLUDE_BUILTINS:
    common.append("--include-builtins")
if INCLUDE_RETURNS:
    common.append("--include-returns")
run([sys.executable, "-u", prepare, "--stage", "metadata", *common])


## Clone repositories and build interprocedural slices


In [ ]:
run([
    sys.executable, "-u", prepare,
    "--stage", "slice",
    *common,
    "--shard-count", SHARD_COUNT,
    "--shard-index", SHARD_INDEX,
    "--slice-log-every", SLICE_LOG_EVERY,
    "--slice-trace-every", SLICE_TRACE_EVERY,
    "--slice-annotation-timeout-seconds", SLICE_ANNOTATION_TIMEOUT_SECONDS,
    "--package-download-timeout-seconds", PACKAGE_DOWNLOAD_TIMEOUT_SECONDS,
    "--kb-phase-timeout-seconds", KB_PHASE_TIMEOUT_SECONDS,
    "--project-analysis-timeout-seconds", PROJECT_ANALYSIS_TIMEOUT_SECONDS,
    "--slice-index-timeout-seconds", SLICE_INDEX_TIMEOUT_SECONDS,
    "--clone-timeout-seconds", CLONE_TIMEOUT_SECONDS,
    "--retrieval-schema-version", RETRIEVAL_SCHEMA_VERSION,
    "--build-import-kb",
    "--download-missing-imports",
    "--kb-max-files-per-package", 3000,
])


## Verify that this shard attempted every assigned project


In [ ]:
import json
sys.path.insert(0, str(PIPELINE_DIR))
from prepare_dataset import project_from_row, read_json, stable_number

projects = set()
for split in ("train", "validation", "test"):
    for row in read_json(WORK_DIR / "metadata" / f"{split}.json"):
        projects.add(project_from_row(row))
selected = {
    project for project in projects
    if stable_number(project, SEED + 4) % SHARD_COUNT == SHARD_INDEX
}
statuses = []
for path in (WORK_DIR / "project_status").glob("*.json"):
    statuses.append(json.loads(path.read_text(encoding="utf-8")))
attempted = {item.get("project") for item in statuses}
missing = sorted(selected - attempted)
summary = {
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "selected_projects": len(selected),
    "attempted_projects": len(selected & attempted),
    "successful_projects": sum(item.get("project") in selected and "error" not in item for item in statuses),
    "failed_projects": sum(item.get("project") in selected and "error" in item for item in statuses),
    "exported_slices": sum(int(item.get("exported", 0)) for item in statuses if item.get("project") in selected),
    "missing_projects": missing,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
(WORK_DIR / "shard_manifest.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
if missing:
    raise RuntimeError(f"Shard is incomplete: {len(missing)} projects missing")


## Package and publish this shard as a public Kaggle Dataset


In [ ]:
publish_command = [
    sys.executable, "-u", PIPELINE_DIR / "publish_shard.py",
    "--work-dir", WORK_DIR,
    "--payload-dir", PUBLISH_DIR,
    "--dataset-id", dataset_id,
    "--title", f"TypePro Python shard {SHARD_INDEX:02d} of {SHARD_COUNT}",
    "--message", f"Completed TypePro shard {SHARD_INDEX:02d} of {SHARD_COUNT}",
    "--expected-shard-index", SHARD_INDEX,
    "--expected-shard-count", SHARD_COUNT,
]
if PUBLISH_PUBLIC:
    publish_command.append("--public")
run(publish_command)
